In [47]:
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer


df = pd.read_csv("Dataset.csv")
print(df.head())
print(df.shape)
print(df.columns)
print(df.info())
print(df.duplicated())
print(df.isnull().sum())



  show_id     type                             title         director  \
0      s1    Movie              Dick Johnson Is Dead  Kirsten Johnson   
1      s3  TV Show                         Ganglands  Julien Leclercq   
2      s6  TV Show                     Midnight Mass    Mike Flanagan   
3     s14    Movie  Confessions of an Invisible Girl    Bruno Garotti   
4      s8    Movie                           Sankofa     Haile Gerima   

         country date_added  release_year rating  duration  \
0  United States  9/25/2021          2020  PG-13    90 min   
1         France  9/24/2021          2021  TV-MA  1 Season   
2  United States  9/24/2021          2021  TV-MA  1 Season   
3         Brazil  9/22/2021          2021  TV-PG    91 min   
4  United States  9/24/2021          1993  TV-MA   125 min   

                                           listed_in  
0                                      Documentaries  
1  Crime TV Shows, International TV Shows, TV Act...  
2                 TV Dr

In [22]:
df['content'] = (
    df['type'] + ' ' +
    df['director'] + ' ' +
    df['country'] + ' ' +
    df['listed_in']
)
df['content'] = df['content'].str.lower()


print(df[['title', 'content']].head())

tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['content'])
print(tfidf_matrix.shape)

                              title  \
0              Dick Johnson Is Dead   
1                         Ganglands   
2                     Midnight Mass   
3  Confessions of an Invisible Girl   
4                           Sankofa   

                                             content  
0  movie kirsten johnson united states documentaries  
1  tv show julien leclercq france crime tv shows,...  
2  tv show mike flanagan united states tv dramas,...  
3  movie bruno garotti brazil children & family m...  
4  movie haile gerima united states dramas, indep...  
(8790, 6559)


In [18]:


df['content'] = df['content'].apply(
    lambda x: re.sub(r"[^\w\s]", " ", x)
)
df['content'] = df['content'].str.replace(
    r"\s+", " ", regex=True
).str.strip()
print(df[['title', 'content']].head())

                              title  \
0              Dick Johnson Is Dead   
1                         Ganglands   
2                     Midnight Mass   
3  Confessions of an Invisible Girl   
4                           Sankofa   

                                             content  
0  movie kirsten johnson united states documentaries  
1  tv show julien leclercq france crime tv shows ...  
2  tv show mike flanagan united states tv dramas ...  
3  movie bruno garotti brazil children family mov...  
4  movie haile gerima united states dramas indepe...  


In [24]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(tfidf_matrix)
print(similarity_matrix.shape)

(8790, 8790)


In [ ]:
import pandas as pd
def recommend(title, n=5):

  idx = df[df['title'].str.lower() == title.lower()].index[0]

  scores = list(enumerate(similarity_matrix[idx]))

  scores = sorted(scores, key=lambda x: x[1], reverse=True)

  scores = scores[1:n+1]

  recommendations = [df.iloc[i]['title'] for i, score in scores]

  return recoommendations

  print(recommend('Ganglands'))

In [36]:
print(recommend('Ganglands'))


Recommendations for: Ganglands

Sentinelle - Similarity: 0.759
Lupin - Similarity: 0.730
Crime Time - Similarity: 0.730
Earth and Blood - Similarity: 0.688
Taken - Similarity: 0.677
None


In [37]:
test_titles = [
    'Ganglands',
    'Midnight Mass',
    'Sankofa',
    'Dick Johnson Is Dead'
]

for title in test_titles:
    print('\n' + '=' * 50)
    recommend(title)



Recommendations for: Ganglands

Sentinelle - Similarity: 0.759
Lupin - Similarity: 0.730
Crime Time - Similarity: 0.730
Earth and Blood - Similarity: 0.688
Taken - Similarity: 0.677


Recommendations for: Midnight Mass

Brand New Cherry Flavor - Similarity: 0.728
The Haunting of Bly Manor - Similarity: 0.728
Ratched - Similarity: 0.728
The Haunting of Hill House - Similarity: 0.728
The Originals - Similarity: 0.728


Recommendations for: Sankofa

Residue - Similarity: 0.550
Lady-Like - Similarity: 0.317
Adhugo - Similarity: 0.281
The Darkest Dawn - Similarity: 0.262
Yeh Hai Bakrapur - Similarity: 0.255


Recommendations for: Dick Johnson Is Dead

The Debt Collector 2 - Similarity: 0.393
Carrie Pilby - Similarity: 0.390
Alex Strangelove - Similarity: 0.387
S.W.A.T. - Similarity: 0.376
Kung Fu Panda: Holiday - Similarity: 0.375


In [48]:
def get_recommendations(title, n=5):
    matches = df[df['title'].str.lower() == title.lower()]

    if matches.empty:
        return[]
    
    idx = matches.index[0]

    scores = list(enumerate(similarity_matrix[idx]))

    scores = sorted(
        scores,
        key=lambda x: x[1],
        reverse=True
    )

    scores = scores[1:n+1]
    return[i for i, score in scores]

In [50]:
def precision_at_k(title, k=5):
    matches = df[df['title'].str.lower() == title.lower()]

    if matches.empty:
        return 0
    
    original_idx = matches.index[0]

    original_genres = set(
        df.loc[original_idx, 'listed_in'].lower().split(", ")
    )

    recommended_indices = get_recommendations(title, k)

    relevant = 0

    for idx in recommended_indices:
        recommended_genres = set(
            df.loc[idx, 'listed_in'].lower().split(", ")
        )

        if original_genres.intersection(recommended_genres):
            relevant += 1
    
    precision = relevant / k

    return precision

    score = precision_at_k('Ganglands', 5)

    print('Precision@5:', score)
    print('Percentage:', score * 100, '%')


In [49]:
test_titles = [
    'Ganglands',
    'Midnight Mass',
    'Sankofa',
    'Dick Johnson Is Dead'
]

for title in test_titles:
    score = precision_at_k(title, 5)


    print(
        title,
        "→ Precision@5:",
        round(score * 100, 2),
        '%'
    )

Ganglands → Precision@5: 60.0 %
Midnight Mass → Precision@5: 100.0 %
Sankofa → Precision@5: 100.0 %
Dick Johnson Is Dead → Precision@5: 0.0 %


In [51]:
df.to_csv('Cleaned_Dataset.csv')